In [1]:
import torch
import pandas as pd
import os
import time
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, models
from torchvision.transforms import v2
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# Use GPU's if avaliable
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
# Test Dataset
# https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html
class TestDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.filenames = sorted(
            file for file in os.listdir(root) if file.lower().endswith(('.png', '.jpg', '.jpeg'))
        )
        self.filepaths = [os.path.join(root, file) for file in self.filenames]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        path = self.filepaths[idx]
        img = Image.open(path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        filename = self.filenames[idx]
        return img, filename

In [4]:
# https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html
data_root = "/kaggle/input/msu-cse-404-fs-25-project/medical_image_dataset"
train_dir = os.path.join(data_root, "train")
val_dir = os.path.join(data_root, "validation")
test_dir = os.path.join(data_root, "test")

# Stats from our training set
stats_mean = (0.2739, 0.2251, 0.2660)
stats_std  = (0.2927, 0.2269, 0.2802)

# Image Transforms (resizing, augmentations)
# https://docs.pytorch.org/vision/stable/transforms.html
image_size = 224 
train_transform = v2.Compose([
    v2.Resize((image_size, image_size)),
    v2.RandomHorizontalFlip(), # defaults to 50% will be horizontal flipped
    v2.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1), # Additional color jitter augmentation (Parameters = +-additional percentage)
    v2.RandomRotation(10), # Additional rotation augmentation (Parameter = +-degrees), might delete since this operation takes longer than most
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=stats_mean, std=stats_std)
])

eval_transform = v2.Compose([
    v2.Resize((image_size, image_size)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=stats_mean, std=stats_std)
])


# Train and Validation datasets
# https://docs.pytorch.org/vision/main/generated/torchvision.datasets.ImageFolder.html
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=eval_transform)
test_dataset = TestDataset(test_dir, transform=eval_transform)

# 
use_subset = False          # Set to False if ready to train with full dataset
subset_fraction = 0.1      # Fraction of the full dataset used

if use_subset:
    num_subset = int(len(train_dataset) * subset_fraction)

    # Random subset of indices
    subset_indices = torch.randperm(len(train_dataset))[:num_subset]

    # Wrap the original dataset
    train_dataset_subset = Subset(train_dataset, subset_indices)
else:
    train_dataset_subset = train_dataset

# Check it works
num_classes = len(train_dataset.classes)
print("Classes: ", train_dataset.classes)
print("num_classes: ", num_classes)

Classes:  ['0428CB', '0BB10A', '14C860', '181D3F', '1BE6F1', '2824E0', '3A8EF1', '3BA5CE', '3F6B3A', '42700D', '94A0EA', '99A880', 'B0B47A', 'C93963', 'CDF490', 'CEC5E4', 'D811C1', 'DF6E49', 'E6CA09', 'F0C556', 'F3602D']
num_classes:  21


In [5]:
# Dataloaders 
# https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html
batch_size = 64

train_loader = DataLoader(train_dataset_subset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True) # Added a optional subset training
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

In [6]:
def get_model(num_classes):
    # Load ResNet-18
    model = models.resnet18(weights='DEFAULT')
    
    # Replace the final fully-connected layer
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    
    return model

model = get_model(num_classes).to(device)
loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4) # AdamW for transfer learning
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1) # lr scheduler to reduce lr during training. step_size = the occurence of epochs before reduction. gamma = reduce amount %

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 183MB/s]


In [7]:
# Train loop
def train_single_epoch(model, loader, optimizer, loss_func, device):
    model.train() # train mode
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)  # use GPU for train
        labels = labels.to(device)

        optimizer.zero_grad() # set to 0 before doing backprop
        outputs = model(images)
        loss = loss_func(outputs, labels)
        loss.backward() # backprop
        optimizer.step() # adam step

        running_loss += loss.item() * images.size(0) # add batch loss scaled by batch size
        _, preds = outputs.max(1) # get predicted class with highest val
        correct += (preds == labels).sum().item() # count how many predictions were right
        total += labels.size(0) # num. of samples in this batch

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# disable gradient computation for speed up
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval() # eval mode 
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device) 
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [8]:
# Train loop
num_epochs = 6

for epoch in range(num_epochs):
    start = time.time()
    train_loss, train_acc = train_single_epoch(model, train_loader, optimizer, loss_func, device)
    val_loss, val_acc = evaluate(model, val_loader, loss_func, device)

    scheduler.step()
    
    end = time.time()
    
    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss} | Train Acc: {train_acc}")
    print(f"Val Loss: {val_loss} | Val Acc: {val_acc}")
    print(f"Time: {end - start} sec")

Epoch 1
Train Loss: 0.6947676448375091 | Train Acc: 0.7490142018031432
Val Loss: 0.5907933134298219 | Val Acc: 0.7863105666877697
Time: 1381.2858319282532 sec
Epoch 2
Train Loss: 0.5815795542709447 | Train Acc: 0.7891559043791522
Val Loss: 0.5563120509487565 | Val Acc: 0.7974558678464687
Time: 1208.101238489151 sec
Epoch 3
Train Loss: 0.5430917863916493 | Train Acc: 0.8027961968694204
Val Loss: 0.5403150583801704 | Val Acc: 0.8053768348522549
Time: 1220.455598115921 sec
Epoch 4
Train Loss: 0.4666376815307867 | Train Acc: 0.8320231418098308
Val Loss: 0.4843936639307574 | Val Acc: 0.8260773545736959
Time: 1210.7697451114655 sec
Epoch 5
Train Loss: 0.44733196488314303 | Train Acc: 0.8385751509454199
Val Loss: 0.4784232972996068 | Val Acc: 0.8301408989855862
Time: 1213.62926030159 sec
Epoch 6
Train Loss: 0.4350849327633451 | Train Acc: 0.8428779629150605
Val Loss: 0.4772782811737642 | Val Acc: 0.8295961484666009
Time: 1207.5386805534363 sec


In [9]:
def create_submission(model, loader, device, class_names):
    model.eval() # Set model to evaluation mode
    
    all_filenames = []
    all_predictions = []
    
    for images, filenames in loader:
        images = images.to(device)
        
        # Get raw model outputs
        outputs = model(images)
        
        # Get the index of the highest score
        _, preds = torch.max(outputs, 1)
        
        # Convert indices back to the class strings
        # ImageLoader turns folder names into indices, but we can convert back using it
        predicted_labels = [class_names[p] for p in preds.cpu().numpy()]
        
        all_filenames.extend(filenames)
        all_predictions.extend(predicted_labels)
        
    submission_df = pd.DataFrame({
        'filename': all_filenames,
        'label': all_predictions
    })
    
    submission_df.to_csv('submission.csv', index=False)
    
    return submission_df
    
# train_dataset.classes contains the mapping for index to folder name
submission_df = create_submission(model, test_loader, device, train_dataset.classes)

Trained with subset (10% of total data after augmentation) (Eddie Lim)
0.75161 (First test)

Val Acc: 0.7632396460593925 lr: 0.001 -> 0.0001

Val Acc: 0.7662873043683103 added more data augmentations

Tried train above 6 epochs to see if there is potential val acc. It fluctuates around 5 to 6 epochs meaning convergence is around that range.

Val Acc: 0.7818642246138897 Added a lr scheduler since it was obeserved that the val acc was oscillating around epoch 5-6. Reducing the step size over time might help